In [64]:
# Import pandas for working with data in tabular format.
import pandas as pd

# Import NumPy for numerical operations and array manipulation.
import numpy as np

# Import train_test_split to divide the dataset into training and testing sets.
from sklearn.model_selection import train_test_split

# Import StandardScaler to standardize the features before applying PCA.
from sklearn.preprocessing import StandardScaler

# Import PCA (Principal Component Analysis) for reducing the number of features while retaining as much important information as possible.
from sklearn.decomposition import PCA

# Import Logistic Regression for building the classification model.
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics to measure the model's performance,
# generate a detailed classification report, and display the confusion matrix.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [65]:

# -----------------------------------------------------------------------------
# Load the Breast Cancer Dataset
# ----------------------------------------------------------------------------- 

# Load the dataset from a CSV file into a Pandas DataFrame.
df = pd.read_csv("assets/breast_cancer_dataset.csv")

print("First 5 rows of the dataset:")
print(df.head())

First 5 rows of the dataset:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter 

In [66]:
# -----------------------------------------------------------------------------
# Separate Features and Target Variable
# -----------------------------------------------------------------------------

# Separate the input features (x) from the target variable (y).
# The 'target' column is the target variable that the model will predict.
X = df.drop('target', axis=1)
# The output is a binary classification (0 = Benign - Not a Cancer, 1 = Malignant - A Cancer)
Y = df['target']

# Display the first five rows of the input features.
print(X.head())

# Display the first five values of the target variable.
print(Y.head())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0           

In [67]:
# -----------------------------------------------------------------------------
# Data Pre-processing - One-Hot Encoding
# -----------------------------------------------------------------------------

# Convert categorical target values into numerical columns using one-hot encoding.
# Convert 'target (Benign/Malignant)' into binary columns.
# This allows the machine learning model to work with categorical target data.
# drop_first=True removes one category from each target to avoid redundancy.
Y = pd.get_dummies(Y, columns=['target'], drop_first=True)

# Display the first five rows of the one-hot encoded features.
print("First 5 rows of the one-hot encoded features:")
print(Y.head())

First 5 rows of the one-hot encoded features:
   malignant
0       True
1       True
2       True
3       True
4       True


In [68]:

# -----------------------------------------------------------------------------
# Split the Dataset into Training and Testing Sets
# -----------------------------------------------------------------------------

# Split the dataset into training and testing sets, with 80% of the data used for training and 20% for testing.
# The random_state parameter ensures reproducibility of the split.
# 42 means that the random number generator will produce the same results each time the code is run.
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}, Y_test shape: {Y_test.shape}")

X_train shape: (455, 30), Y_train shape: (455, 1)
X_test shape: (114, 30), Y_test shape: (114, 1)


In [69]:
# -----------------------------------------------------------------------------
# Feature Scaling / Data Standardization
# -----------------------------------------------------------------------------

# Standardize the feature values so that all features are on a
# similar scale. This is particularly important before applying
# PCA because PCA is sensitive to differences in feature scales.
#
# StandardScaler standardizes each feature by:
#   1. Calculating the mean of the feature.
#   2. Calculating the standard deviation of the feature.
#   3. Transforming the values so they are centered around a
#      mean of 0 with a standard deviation of 1.
#
# IMPORTANT:
# We fit the scaler ONLY on the training data. This means the
# mean and standard deviation are calculated using X_train only.
#
# The same scaling parameters learned from the training data are
# then applied to both the training and testing datasets.
#
# We do NOT use fit_transform() on X_test because calculating a
# separate mean and standard deviation from the testing data would
# cause data leakage. The testing data should remain completely
# unseen during the training process.

# Create an instance of the StandardScaler.
scaler = StandardScaler()

# Calculate the mean and standard deviation from the training data,
# then use them to standardize the training features.
X_train_scaled = scaler.fit_transform(X_train)

# Apply the SAME mean and standard deviation obtained from the training data to standardize the testing features.
X_test_scaled = scaler.transform(X_test)

In [70]:
# -----------------------------------------------------------------------------
# Apply Principle Component Analysis (PCA)
# -----------------------------------------------------------------------------

# PCA is used to reduce the dimensionality of the dataset while
# retaining as much important information (variance) as possible.
#
# Instead of manually specifying the exact number of principal
# components, n_components=0.90 instructs PCA to automatically
# select the minimum number of components required to preserve
# at least 90% of the total variance in the original dataset.
#
# IMPORTANT:
# PCA is fitted ONLY on the scaled training data. During fitting,
# PCA learns the directions (principal components) that capture
# the maximum variance from the training dataset.
#
# The learned PCA transformation is then applied to the testing
# data using transform() only. We do not fit PCA separately on
# the testing data because this would introduce data leakage.

# Create a PCA object that automatically retains at least 90% of the variance in the training dataset.
pca_90 = PCA(n_components=0.90)

# Learn the principal components from the scaled training data and transform the training data into the reduced PCA space.
X_train_pca = pca_90.fit_transform(X_train_scaled)

# Apply the SAME PCA transformation learned from the training data to transform the scaled testing data.
X_test_pca = pca_90.transform(X_test_scaled)

# Display the shape of the transformed training dataset and the exact number of principal components selected by PCA.
print(
    f"PCA Training Shape: {X_train_pca.shape} "
    f"(We kept exactly {pca_90.n_components_} components!)"
)

PCA Training Shape: (455, 7) (We kept exactly 7 components!)


In [71]:
# -----------------------------------------------------------------------------
# Train and Evaluate The LOGISTIC REGRESSION MODEL
# -----------------------------------------------------------------------------

# Train a Logistic Regression classification model using the
# reduced feature set generated by PCA.
#
# Instead of using all the original features, the model is trained
# using only the selected principal components that preserve at
# least 90% of the variance in the original training dataset.
#
# After training, the model is used to make predictions on the
# PCA-transformed testing data. The predictions are then compared
# with the actual class labels to evaluate the model's performance.

# Create an instance of the Logistic Regression classification model.
model = LogisticRegression()

# Train the model using the PCA-reduced training features and their corresponding target labels.
# to_numpy() converts the Pandas DataFrame into a NumPy array,
# while ravel() converts the target values into a 1D array, which is the format expected by Logistic Regression.
model.fit(X_train_pca, Y_train.to_numpy().ravel())

# Use the trained model to predict the class labels for the PCA-transformed testing dataset.
Y_test_pred = model.predict(X_test_pca)

# Calculate the accuracy by comparing the predicted labels with the actual labels from the testing dataset.
accuracy = accuracy_score(Y_test, Y_test_pred)

# Display the final model accuracy and the number of principal components used after dimensionality reduction.
print(f"Model Accuracy using only {pca_90.n_components_} components: {accuracy * 100:.2f}%")

Model Accuracy using only 7 components: 98.25%


In [72]:
# -----------------------------------------------------------------------------
# Generate the Confusion Matrix to Evaluate the Classification Results
# -----------------------------------------------------------------------------

confusion_mat = confusion_matrix(Y_test, Y_test_pred)

# Generate the classification report containing precision, recall, F1-score, and support (means the count) for each sentiment class (-1 and +1).
classification_repo = classification_report(Y_test, Y_test_pred)

# Display the confusion matrix.
print("Confusion Matrix:")
print(confusion_mat)

# Display the classification report.
print("Classification Report:")
print(classification_repo)

Confusion Matrix:
[[70  1]
 [ 1 42]]
Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.99      0.99        71
        True       0.98      0.98      0.98        43

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [79]:
# -----------------------------------------------------------------------------
# Predict Unknown Data Samples
# -----------------------------------------------------------------------------

# Create two unknown data samples.
# Each sample contains values for the same 30 features used to train the model.
unknown_data = np.array([
    [
        17.99, 10.38, 122.80, 1001.0, 0.11840,
        0.27760, 0.3001, 0.14710, 0.2419, 0.07871,
        1.0950, 0.9053, 8.5890, 153.40, 0.006399,
        0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
        25.38, 17.33, 184.60, 2019.0, 0.1622,
        0.6656, 0.7119, 0.2654, 0.4601, 0.11890
    ],
    [
        12.0, 15.0, 78.0, 450.0, 0.090,
        0.100, 0.080, 0.050, 0.180, 0.060,
        0.300, 0.700, 2.000, 25.0, 0.006,
        0.030, 0.040, 0.010, 0.020, 0.005,
        13.5, 18.0, 85.0, 550.0, 0.120,
        0.200, 0.250, 0.100, 0.250, 0.070
    ]
])

# Convert the unknown data into a Pandas DataFrame using the
# same feature names and order as the training data.
unknown_data_df = pd.DataFrame(unknown_data, columns=X.columns)

# Standardize the Unknown Data.
# The unknown samples must be standardized using the SAME scaler
# that was fitted on the training data.
#
# We use transform() rather than fit_transform() because we must not
# calculate new mean and standard deviation values from the unknown data.
#
# This ensures that the unknown samples are represented on the same
# scale as the data used to train the model.
unknown_data_scaled = scaler.transform(unknown_data_df)


# Apply PCA Transformation to the Unknown Data.
# Apply the SAME PCA transformation that was learned from the training data.
#
# We use transform() rather than fit_transform() because PCA has already
# learned the principal components from the training data.
#
# The 30 original features are therefore transformed into the same
# 7 principal components used by the Logistic Regression model.
unknown_data_pca = pca_90.transform(unknown_data_scaled)

# Display the shape of the transformed unknown data.
print("Unknown data PCA shape:", unknown_data_pca.shape)


# Predict the Class of the Unknown Samples and Display the Predictions.
# Use the trained Logistic Regression model to predict the class of each unknown sample.
#
# The model returns the encoded class labels:
# False = Benign
# True  = Malignant
unknown_predictions = model.predict(unknown_data_pca)

# Display the predicted class for each unknown sample.
for i, prediction in enumerate(unknown_predictions, start=1):
    # Convert the Boolean prediction into a meaningful medical class label.
    if prediction:
        result = "Malignant"
    else:
        result = "Benign"

    print(f"Unknown Sample {i}: {result}")

Unknown data PCA shape: (2, 7)
Unknown Sample 1: Malignant
Unknown Sample 2: Benign
